# EX02: Language Processing with `Ply` and `Lark`

## Grammar Engineering

## MEI/2026-27

#### Nuno Macedo
Universidade do Minho

The following programming exercises are intended to be solved with two parser generator modules for Python, [Ply](https://ply.readthedocs.io/) and [Lark](https://lark-parser.readthedocs.io/).

# Module [`Ply`](https://ply.readthedocs.io/) basics

- A Python implementation of the popular lex/yacc compiler front-end tools

- The scanning and parsing phases are distinct:
  - The lexer definition is given as a set of regular expressions that represent different token types
  - The parser definition is given as a grammar in BNF whose terminal symbols are token types

- It relies on a *translating grammar*, a grammar enhanced with semantic actions that drive a *syntax-directed translation*

- This makes it easier to write simple direct translations, makes the translation tightly coupled to the grammar
  - Less suitable for multiple translations
  - Harder to understand the semantics
  - Less prone to formal analysis

- Supports LALR(1) parsing (Look-Ahead Left-to-right Rightmost derivation with 1 token of lookahead)

- In traditional lex/yacc, the specifications are given as separate files and compiled into a target language
  - In `Ply` they are written directly in Python (uses reflection)


In [2]:
!pip install --upgrade pip
!pip install Ply

⚠️ *Note:* since `Ply` relies on reflection, it is not prone to having multiple specifications in the same module
  - This leads to some conflicts when writing specifications in notebooks
  - Either run outside the notebook, or use the following function to clean tokens


In [3]:
__file__ = "Untitled.ipynb" # needed to run Ply inside notebook

def cleanLexerGlobals():
  "Remove ply.lex identifiers from the global context."
  for t in [n for n in globals() if n.startswith("t_") or n in ("tokens", "literals", "states")]:
    del globals()[t]

def cleanParserGlobals():
  "Remove ply.yacc identifiers from the global context."
  for t in [n for n in globals() if n.startswith("p_")]:
    del globals()[t]
  import os
  try: # avoid conflicts in temp files when running from notebook
    os.remove("parsetab.py")
  except FileNotFoundError:
    pass

## [`Ply.lex`](https://ply.readthedocs.io/en/latest/ply.html#lex) basics

- The list of token types is given in a variable `tokens`

- After scanning, each token has a *type* and a *value* assigned (by default, the lexeme)

- Each token type `T` is defined by a variable `t_T` that is assigned a regular expression
  - Either directly a string with a RE
  - Or a function that has a RE as docstring and further actions

- Can also define single-symbol literal tokens in variable *literals* (both type and value become the symbol)

- The master RE is built in the following order:
  - Function-defined tokens by order they appear
  - Literal-defined tokens by descending length

- Other special function:
  - `t_ignore` to automatically ignore certain symbols
  - `t_error` to handle errors

## Example: Simple arithmetic expressions

In [5]:
#cleanLexerGlobals()
#cleanParserGlobals()

import ply.lex as lex
import math

tokens = (
  'INT', 'FLOAT', 'PLUS', 'MINUS', 'DIVIDE', 'TIMES'
)

literals = ('(', ')')

t_ignore = ' \t\n'
t_PLUS    = r'\+'
t_MINUS   = r'-'
t_TIMES   = r'\*'
t_DIVIDE  = r'/'

def t_FLOAT(t):
  r'\d+\.\d*|\.\d+'
  t.value = float(t.value)
  return t

def t_INT(t):
  r'\d+'
  t.value = int(t.value)
  return t

def t_error(t):
  print(f"Illegal character {t.value[0]}")
  t.lexer.skip(1)


#__file__ = "Untitled.ipynb" # needed to run PLY inside notebook
lexer = lex.lex()

In [9]:
def run_lexer(data):
    lexer.input(data)
    return [(tok.type, tok.value) for tok in lexer]

print(run_lexer("10+20"))
print(run_lexer("(10.23 + 20) * 293"))
print(run_lexer("10-20"))
print(run_lexer("10.23 + 20 * -293"))
print(run_lexer("10 - -20"))

[('INT', 10), ('PLUS', '+'), ('INT', 20)]
[('(', '('), ('FLOAT', 10.23), ('PLUS', '+'), ('INT', 20), (')', ')'), ('TIMES', '*'), ('INT', 293)]
[('INT', 10), ('MINUS', '-'), ('INT', 20)]
[('FLOAT', 10.23), ('PLUS', '+'), ('INT', 20), ('TIMES', '*'), ('MINUS', '-'), ('INT', 293)]
[('INT', 10), ('MINUS', '-'), ('MINUS', '-'), ('INT', 20)]


## [`Ply.yacc`](https://ply.readthedocs.io/en/latest/ply.html#yacc) basics

- Rules are given by special functions starting with `p_`
  - You can write a different function for each production, or merge them into a single function

- The grammar rules are given in the docstring in BNF

- The starting grammar symbol is the one defined first

- The terminals of the grammar are the token types of the lexer
   - They may not be constant: terminal `INT` is any integer number

- Each variable is assigned a *value* during parsing, defined by the semantic actions

- The semantic actions are executed in reduce steps
  - use the values of the sub-expressions (obtained through the appropriate index)
  - updates the value of variable being expanded

- For tokens (terminals) the value is that assigned during lexing

- `precedence` of operators can be given to disambiguate grammars

In [10]:
import ply.yacc as yacc

import os
try: # avoid conflicts in temp files when running from notebook
    os.remove("parsetab.py")
except FileNotFoundError:
    pass

def p_expr_add_sub(p):
    '''expr : expr PLUS term
            | expr MINUS term'''
    if p[2] == '+':
        p[0] = p[1] + p[3]
    else:
        p[0] = p[1] - p[3]

def p_expr_term(p):
    'expr : term'
    p[0] = p[1]

def p_term_mul_div(p):
    '''term : term TIMES factor
            | term DIVIDE factor'''
    if p[2] == '*':
        p[0] = p[1] * p[3]
    else:
        p[0] = p[1] / p[3]

def p_term_factor(p):
    'term : factor'
    p[0] = p[1]

def p_factor_int(p):
    'factor : INT'
    p[0] = p[1]

def p_factor_float(p):
    'factor : FLOAT'
    p[0] = p[1]

def p_factor_parens(p):
    'factor : "(" expr ")"'
    p[0] = p[2]

def p_factor_uminus(p):
    'factor : MINUS factor'
    p[0] = -p[2]

def p_error(p):
    if p:
        print(f"Syntax error at '{p.value}'")
    else:
        print("Syntax error at EOF")

parser = yacc.yacc(write_tables=False, debug=True)

Generating LALR tables


In [11]:
print(parser.parse("(-3.4 * 30) + 20 + -30 * 100"))
print(parser.parse("(10 + 30 + 100) * (1 - 1)"))

-3082.0
0


# Exercise 1

Propositional logic formulas have the following syntax:

- Variables $A$, $B$, $C$, ...
- Conjunction $f_1 ∧ f_2$,
- Disjunction $f_1 ∨ f_2$,
- Negation $¬ f_1$,
- Implication $f_1 → f_2$
- Equivalence $f_1 ↔︎ f_2$
- Constants false $⊥$ and true $⊤$

## Exercise 1.1

Write a lexer using `Ply.lex` to tokenize propositional logic formulas.

In [7]:
import ply.lex as lex
import ply.yacc as yacc
import doctest

In [18]:
cleanLexerGlobals()
cleanParserGlobals()

tokens = (
    'ID', 'AND', 'OR', 'NOT', 'IMPLIES', 'EQ', 'TRUE', 'FALSE'
)

literals = ('(', ')')

t_ignore = ' \t\n'
t_AND     = r'∧'
t_OR      = r'∨'
t_NOT     = r'¬'
t_IMPLIES = r'→'
t_EQ      = r'↔︎'
t_TRUE    = r'⊤'
t_FALSE   = r'⊥'

def t_ID(t):
    r'[a-zA-Z_][a-zA-Z0-9_]*'
    return t

def t_error(t):
    print(f"Illegal character {t.value[0]}")
    t.lexer.skip(1)

lexer = lex.lex()

def run_lexer(data):
    """
    PLY lexer with doctests.

    >>> print([(tok.type, tok.value) for tok in run_lexer("A ∧ ¬B")])
    [('ID', 'A'), ('AND', '∧'), ('NOT', '¬'), ('ID', 'B')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("A → (¬B ∨ C)")])
    [('ID', 'A'), ('IMPLIES', '→'), ('(', '('), ('NOT', '¬'), ('ID', 'B'), ('OR', '∨'), ('ID', 'C'), (')', ')')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("(A → AB) ∧ ¬(¬B ∨ C)")])
    [('(', '('), ('ID', 'A'), ('IMPLIES', '→'), ('ID', 'AB'), (')', ')'), ('AND', '∧'), ('NOT', '¬'), ('(', '('), ('NOT', '¬'), ('ID', 'B'), ('OR', '∨'), ('ID', 'C'), (')', ')')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("⊥ ∧ aux")])
    [('FALSE', '⊥'), ('AND', '∧'), ('ID', 'aux')]
    """
    lexer.input(data)
    return lexer

doctest.run_docstring_examples(run_lexer, globals())

## Exercise 1.2

Write a parser using `Ply.yacc` to parse and retrieve all propositional variables from a propositional formula.

In [19]:
precedence = (
    ('left', 'EQ'),
    ('left', 'IMPLIES'),
    ('left', 'OR'),
    ('left', 'AND'),
    ('right', 'NOT')
)

def p_formula_binop(p):
    '''formula : formula AND formula
               | formula OR formula
               | formula IMPLIES formula
               | formula EQ formula'''
    p[0] = p[1].union(p[3])

def p_formula_not(p):
    'formula : NOT formula'
    p[0] = p[2]

def p_formula_group(p):
    "formula : '(' formula ')'"
    p[0] = p[2]

def p_formula_id(p):
    'formula : ID'
    p[0] = {p[1]}

def p_formula_const(p):
    '''formula : TRUE
               | FALSE'''
    p[0] = set()

def p_error(p):
    if p:
        print(f"Syntax error at '{p.value}'")
    else:
        print("Syntax error at EOF")

def run_parser(data):
  """
  PLY parser with doctests.

  >>> print(sorted(list(run_parser("Chove ∧ ¬Frio"))))
  ['Chove', 'Frio']
  >>> print(sorted(list(run_parser("Dia ∨ ¬Dia"))))
  ['Dia']
  >>> print(sorted(list(run_parser("A ∧ ¬B ∨ ⊤"))))
  ['A', 'B']
  >>> print(sorted(list(run_parser("Reprovar → (¬Presenca ∨ Negativa)"))))
  ['Negativa', 'Presenca', 'Reprovar']
  >>> print(sorted(list(run_parser("A ∧ (X ∧ ¬Y) ∨ ⊤"))))
  ['A', 'X', 'Y']
  """
  return yacc.yacc(write_tables=False, debug=True).parse(data)

doctest.run_docstring_examples(run_parser, globals())

Generating LALR tables
Generating LALR tables
Generating LALR tables
Generating LALR tables
Generating LALR tables


## Exercise 1.3

Write a parser using `Ply.yacc` to parse and evaluate the truth value of a propositional formula.

In [21]:
cleanParserGlobals()

precedence = (
    ('left', 'EQ'),
    ('left', 'IMPLIES'),
    ('left', 'OR'),
    ('left', 'AND'),
    ('right', 'NOT')
)

def p_formula_and(p):
    'formula : formula AND formula'
    p[0] = p[1] and p[3]

def p_formula_or(p):
    'formula : formula OR formula'
    p[0] = p[1] or p[3]

def p_formula_implies(p):
    'formula : formula IMPLIES formula'
    # A -> B é equivalente a não A ou B
    p[0] = (not p[1]) or p[3]

def p_formula_eq(p):
    'formula : formula EQ formula'
    p[0] = p[1] == p[3]

def p_formula_not(p):
    'formula : NOT formula'
    p[0] = not p[2]

def p_formula_group(p):
    "formula : '(' formula ')'"
    p[0] = p[2]

def p_formula_id(p):
    'formula : ID'
    global assignment
    p[0] = assignment.get(p[1], False)

def p_formula_true(p):
    'formula : TRUE'
    p[0] = True

def p_formula_false(p):
    'formula : FALSE'
    p[0] = False

def p_error(p):
    if p:
        print(f"Syntax error at '{p.value}'")
    else:
        print("Syntax error at EOF")

def run_parser(data, asg):
    """
    PLY parser with doctests.

    >>> print(run_parser("Frio ∧ ¬Chove", {"Frio":True, "Chove": True}))
    False
    >>> print(run_parser("Noite ∨ ¬Noite", {"Noite":True, "Dia": False}))
    True
    >>> print(run_parser("A ∧ ¬B ∨ ⊤", {"A":True, "B": True}))
    True
    >>> print(run_parser("Reprovar → (¬Presenca ∨ Negativa)", {"Presenca":True, "Negativa": False, "Reprovar": False}))
    True
    >>> print(run_parser("A ∧ (X ∧ ¬Y)", {"A":True, "X": False, "Y": True}))
    False
    """
    global assignment # not good: hack to run inside notbook
    assignment = asg
    return yacc.yacc(write_tables=False, debug=False).parse(data)

doctest.run_docstring_examples(run_parser, globals())

# Exercise 2

Consider a language to specify monotonic numeric ranges: either always increasing or always decreasing. For instance, the following are valid ranges:
- `[3 ; 3.5] [5.6 ; 12] [-10 ; 2.5]`
- `[3 ; 2.5] [5 ; 1.2] [1 ; -0.8]`

## Exercise 2.1

Write a lexer using `ply.lex` to tokenize strings in this language.

In [8]:
cleanLexerGlobals()
cleanParserGlobals()

tokens = (
    'NUM',
)

literals = ('[', ']', ';')

t_ignore = ' \t\n'

def t_NUM(t):
    r'[-+]?\d+(\.\d+)?'
    t.value = float(t.value)
    return t

def t_error(t):
    print(f"Illegal character '{t.value[0]}'")
    t.lexer.skip(1)

lexer = lex.lex()

def run_lexer(data):
    """
    PLY lexer with doctests.

    >>> print([(tok.type, tok.value) for tok in run_lexer('[-1.5 ; +2] [3 ; +4.7]')])
    [('[', '['), ('NUM', -1.5), (';', ';'), ('NUM', 2.0), (']', ']'), ('[', '['), ('NUM', 3.0), (';', ';'), ('NUM', 4.7), (']', ']')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("[3 ; 3.5] [5 ; 12] [48 ; 73.2]")])
    [('[', '['), ('NUM', 3.0), (';', ';'), ('NUM', 3.5), (']', ']'), ('[', '['), ('NUM', 5.0), (';', ';'), ('NUM', 12.0), (']', ']'), ('[', '['), ('NUM', 48.0), (';', ';'), ('NUM', 73.2), (']', ']')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("[3 ; 3.5] [1 ; 12] [48 ; 73.2]")])
    [('[', '['), ('NUM', 3.0), (';', ';'), ('NUM', 3.5), (']', ']'), ('[', '['), ('NUM', 1.0), (';', ';'), ('NUM', 12.0), (']', ']'), ('[', '['), ('NUM', 48.0), (';', ';'), ('NUM', 73.2), (']', ']')]
    >>> print([(tok.type, tok.value) for tok in run_lexer("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]")])
    [('[', '['), ('NUM', 3.0), (';', ';'), ('NUM', 3.5), (']', ']'), ('[', '['), ('NUM', 2.0), (';', ';'), ('NUM', 1.0), (']', ']'), ('[', '['), ('NUM', 0.5), (';', ';'), ('NUM', -73.2), (']', ']')]
    """

    lexer.input(data)
    return lexer

doctest.run_docstring_examples(run_lexer, globals())

## Exercise 2.2

Write a parser using `ply.yacc` to parse count the number of intervals defined and sum of all intervals.

In [9]:
def p_intervals_multiple(p):
    '''intervals : intervals interval'''
    p[0] = (p[1][0] + p[2][0], round(p[1][1] + p[2][1], 5))

def p_intervals_single(p):
    '''intervals : interval'''
    p[0] = p[1]

def p_interval(p):
    '''interval : '[' NUM ';' NUM ']' '''
    count = 1
    amplitude = p[4] - p[2]
    p[0] = (count, amplitude)

def p_error(p):
    if p:
        print(f"Syntax error at '{p.value}'")
    else:
        print("Syntax error at EOF")

def run_parser(data):
  """
  PLY parser with doctests.

  >>> print(run_parser('[-1 ; 0] [0 ; 1] [2 ; 3] [3 ; 4]'))
  (4, 4.0)
  >>> print(run_parser('[-200 ; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  (4, 182.2)
  >>> print(run_parser("[3 ; 3.5] [5 ; 12] [48 ; 73.2]"))
  (3, 32.7)
  >>> print(run_parser("[34 ; 36.5]"))
  (1, 2.5)
  >>> print(run_parser("[3 ; 3.5] [1 ; 12] [48 ; 73.2]"))
  (3, 36.7)
  >>> print(run_parser("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  (3, -74.2)
  """
  return yacc.yacc(write_tables=False, debug=True).parse(data)

doctest.run_docstring_examples(run_parser, globals())

Generating LALR tables
Generating LALR tables
Generating LALR tables
Generating LALR tables
Generating LALR tables
Generating LALR tables


## Exercise 2.3

Write a parser to test whether the ranges are valid (always decreasing or always increasing).

In [12]:
cleanParserGlobals()

def p_start(p):
    '''start : intervals'''
    p[0] = not ("inc" in p[1] and "dec" in p[1])

def p_intervals_multiple(p):
    '''intervals : intervals interval'''
    p[0] = p[1].union(p[2])

def p_intervals_single(p):
    '''intervals : interval'''
    p[0] = p[1]

def p_interval(p):
    '''interval : '[' NUM ';' NUM ']' '''
    if p[2] < p[4]:
        p[0] = {"inc"}
    elif p[2] > p[4]:
        p[0] = {"dec"}
    else:
        p[0] = set()

def p_error(p):
    if p:
        print(f"Syntax error at '{p.value}'")
    else:
        print("Syntax error at EOF")

def run_parser(data):
  """
  PLY parser with doctests.

  >>> print(run_parser('[-1 ; 0] [0 ; 1] [2 ; 3] [3 ; 4]'))
  True
  >>> print(run_parser('[-200 ; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  True
  >>> print(run_parser("[3 ; 3.5] [5 ; 12] [48 ; 13.2]"))
  False
  >>> print(run_parser("[34 ; 36.5]"))
  True
  >>> print(run_parser("[3 ; 3.5] [63 ; 12] [48 ; 73.2]"))
  False
  >>> print(run_parser("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  False
  """
  return yacc.yacc(write_tables=False, debug=True).parse(data)

doctest.run_docstring_examples(run_parser, globals())

Generating LALR tables
Generating LALR tables
Generating LALR tables
Generating LALR tables
Generating LALR tables
Generating LALR tables


# Module [`Lark`](https://lark-parser.readthedocs.io/) basics

- In `Lark`, the grammar is purely syntactic, there are no semantic actions assigned

- It automatically generates an AST and provides methods to traverse it

- By decoupling the parsing and the transformation, it is more prone to multiple analysis

- No clear distinction between the lexer and the parser, defined in the same specification

- Richer range of parsing algorithms, supports more grammars

In [13]:
!pip install lark

## `Lark` grammar

- Grammar is in extended EBNF
  - operators for repetition of elements `*`, optional elements `?`, ...

- Grammar starting symbol identified with `start`

- Terminal definition is at the same level as the non-terminals
  - Can use regex, identified by `/` delimiters
  - Some predefined terminals can be imported with instruction `%import`

- Unnamed literals in grammar rules ignored in the AST
  - `?` can be used to ignore other variables in the AST (if they have a single parent)

- Rules can be assigned names with `->` to ease writing transformations

- Precedence and associativity rules can be declared with instructions `%left` and `%right`

- `Lark` also supports parsing algorithms that return multiple trees when ambiguous

In [14]:
from lark import Lark, Transformer

grammar = """
start: expr

expr: expr ADD term
    | expr SUB term
    | term

term: term MUL factor
    | term DIV factor
    | factor

factor: NUMBER
      | SUB factor
      | "(" expr ")"

ADD: "+"
SUB: "-"
MUL: "*"
DIV: "/"

NUMBER: /[0-9]+(\\.[0-9]+)?/

%ignore /[ \t\f]+/
"""

parser = Lark(grammar)

In [15]:
print(parser.parse("(-3.4 * 30) + 20 + -30 * 100").pretty())
print(parser.parse("(10 + 30 + 100) * (1 - 1)").pretty())

start
  expr
    expr
      expr
        term
          factor
            expr
              term
                term
                  factor
                    -
                    factor	3.4
                *
                factor	30
      +
      term
        factor	20
    +
    term
      term
        factor
          -
          factor	30
      *
      factor	100

start
  expr
    term
      term
        factor
          expr
            expr
              expr
                term
                  factor	10
              +
              term
                factor	30
            +
            term
              factor	100
      *
      factor
        expr
          expr
            term
              factor	1
          -
          term
            factor	1



## `Lark` transformers

- `Lark` automatically creates an AST for the parsed strings

- Provides helper classes that ease the traversal of these ASTs

- `Transformer` is one such class, performs a bottom-up traversal
  - It provides a methods for each grammar rule, which should be overwritten
  - The argument is a list of values resulting from processing the children nodes

In [16]:
class EvalTransformer(Transformer):
    def start(self, items):
        return items[0]

    def expr(self, items):
        if len(items) == 1:
          return items[0]
        left, op, right = items
        if op.type == 'ADD':
            return left + right
        elif op.type == 'SUB':
            return left - right

    def term(self, items):
        if len(items) == 1:
          return items[0]
        left, op, right = items
        if op.type == 'MUL':
            return left * right
        elif op.type == 'DIV':
            return left / right

    def factor(self, items):
        if len(items) == 1:
            return items[0]
        elif len(items) == 2:
            return -items[1]
        return items[1]

    def NUMBER(self, token):
        return float(token)


In [17]:
print(EvalTransformer().transform(parser.parse("(-3.4 * 30) + 20 + -30 * 100")))
print(EvalTransformer().transform(parser.parse("(10 + 30 + 100) * (1 - 1)")))

-3082.0
0.0


## A bit more advanced version

- Use EBNF to simplify some rules with repetition `*` and alternative operators `|`
  - With `*`, branches can have an arbitrary number of children
  - But with `|` the operator may be different between the arguments

- Use `?` to skip certain rules in the AST with single children
  - Fewer cases to handle in the transformer

In [18]:
from lark import Lark, Transformer

grammar = """
?start: expr

?expr: term ((ADD|SUB) term)*

?term: factor ((MUL|DIV) factor)*

?factor: NUMBER
      | SUB factor
      | "(" expr ")"

ADD: "+"
SUB: "-"
MUL: "*"
DIV: "/"

NUMBER: /[0-9]+(\\.[0-9]+)?/

%ignore /[ \t\f]+/
"""

parser = Lark(grammar)

In [19]:
print(parser.parse("(-3.4 * 30) + 20 + -30 * 100").pretty())
print(parser.parse("(10 + 30 + 100) * (1 - 1)").pretty())

expr
  term
    factor
      -
      3.4
    *
    30
  +
  20
  +
  term
    factor
      -
      30
    *
    100

term
  expr
    10
    +
    30
    +
    100
  *
  expr
    1
    -
    1



In [20]:
class EvalTransformer(Transformer):
    def expr(self, items):
        res = items[0]
        for i in range(1, len(items), 2):
          if items[i].type == 'ADD':
            res += items[i+1]
          elif items[i].type == 'SUB':
            res -= items[i+1]
        return res

    def term(self, items):
        res = items[0]
        for i in range(1, len(items), 2):
          if items[i].type == 'MUL':
            res *= items[i+1]
          elif items[i].type == 'DIV':
            res /= items[i+1]
        return res

    def factor(self, items):
        return -items[1]

    def NUMBER(self, token):
        return float(token)

In [21]:
print(EvalTransformer().transform(parser.parse("(-3.4 * 30) + 20 + -30 * 100")))
print(EvalTransformer().transform(parser.parse("(10 + 30 + 100) * (1 - 1)")))

-3082.0
0.0


# Exercise 3

Recall the language for propositional logic formulas from Exercise 1.

## Exercise 3.1

Write a `Lark` grammar to obtain the AST for propositional logic formulas.

In [23]:
def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser("A ∧ ¬B")
  Tree(Token('RULE', 'formula'), [Tree(Token('RULE', 'implies'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'neg'), [Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'var'), [Token('ID', 'A')])])])]), Token('CONJ', '∧'), Tree(Token('RULE', 'neg'), [Token('NEG', '¬'), Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'var'), [Token('ID', 'B')])])])])])])])

  >>> run_parser("A ∧ ¬B")
  Tree(Token('RULE', 'formula'), [Tree(Token('RULE', 'implies'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'neg'), [Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'var'), [Token('ID', 'A')])])])]), Token('CONJ', '∧'), Tree(Token('RULE', 'neg'), [Token('NEG', '¬'), Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'var'), [Token('ID', 'B')])])])])])])])

  >>> run_parser("A ∧ ¬B ∨ ⊤")
  Tree(Token('RULE', 'formula'), [Tree(Token('RULE', 'implies'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'neg'), [Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'var'), [Token('ID', 'A')])])])]), Token('CONJ', '∧'), Tree(Token('RULE', 'neg'), [Token('NEG', '¬'), Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'var'), [Token('ID', 'B')])])])])]), Token('DISJ', '∨'), Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'neg'), [Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'constant'), [Token('TRUE', '⊤')])])])])])])])

  >>> run_parser("A ∧ ¬B ∨ ⊤")
  Tree(Token('RULE', 'formula'), [Tree(Token('RULE', 'implies'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'neg'), [Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'var'), [Token('ID', 'A')])])])]), Token('CONJ', '∧'), Tree(Token('RULE', 'neg'), [Token('NEG', '¬'), Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'var'), [Token('ID', 'B')])])])])]), Token('DISJ', '∨'), Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'neg'), [Tree(Token('RULE', 'factor'), [Tree(Token('RULE', 'constant'), [Token('TRUE', '⊤')])])])])])])])
  """
  grammar = """
    formula: eq
    ?eq: eq EQ implies | implies
    implies: implies IMPLIES disj | disj
    disj: disj DISJ conj | conj
    conj: conj CONJ neg | neg
    neg: NEG factor | factor
    factor: var | constant | "(" formula ")"
    var: ID
    constant: TRUE | FALSE

    ID: /[a-zA-Z_][a-zA-Z0-9_]*/
    CONJ: "∧"
    DISJ: "∨"
    NEG: "¬"
    IMPLIES: "→"
    EQ: "↔︎"
    TRUE: "⊤"
    FALSE: "⊥"

    %ignore /[ \\t\\n]+/
    """

  parser = Lark(grammar, start='formula')
  return parser.parse(data)

doctest.run_docstring_examples(run_parser, globals())

## Exercise 3.2

Write a `lark` transformer to retrieve all propositional variables of a propositional formula.

In [25]:

def vars(data):
  """
  Lark variable collector with doctests.

  >>> print(sorted(list(vars("A ∧ ¬B"))))
  ['A', 'B']
  >>> print(sorted(list(vars("A ∨ ¬A"))))
  ['A']
  >>> print(sorted(list(vars("A ∧ ¬B ∨ ⊤"))))
  ['A', 'B']
  >>> print(sorted(list(vars("A → (¬B ∨ C)"))))
  ['A', 'B', 'C']
  >>> print(sorted(list(vars("A ∧ (X ∧ ¬Y) ∨ ⊤"))))
  ['A', 'X', 'Y']
  """
  return None

doctest.run_docstring_examples(vars, globals())

**********************************************************************
File "Untitled.ipynb", line 5, in NoName
Failed example:
    print(sorted(list(vars("A ∧ ¬B"))))
Exception raised:
    Traceback (most recent call last):
      File "<doctest NoName[0]>", line 1, in <module>
        print(sorted(list(vars("A ∧ ¬B"))))
                     ~~~~^^^^^^^^^^^^^^^^
    TypeError: 'NoneType' object is not iterable
**********************************************************************
File "Untitled.ipynb", line 7, in NoName
Failed example:
    print(sorted(list(vars("A ∨ ¬A"))))
Exception raised:
    Traceback (most recent call last):
      File "<doctest NoName[1]>", line 1, in <module>
        print(sorted(list(vars("A ∨ ¬A"))))
                     ~~~~^^^^^^^^^^^^^^^^
    TypeError: 'NoneType' object is not iterable
**********************************************************************
File "Untitled.ipynb", line 9, in NoName
Failed example:
    print(sorted(list(vars("A ∧ ¬B ∨ ⊤"))))


## Exercise 3.3

Write a `lark` transformer to evaluate the truth value of a propositional formula.

In [ ]:
def eval(data, assignment):
  """
  Lark evaluator with doctests.

  >>> print(eval("A ∧ ¬B", {"A":True, "B": True}))
  False
  >>> print(eval("A ∨ ¬A", {"A":True, "B": False}))
  True
  >>> print(eval("A ∧ ¬B ∨ ⊤", {"A":True, "B": True}))
  True
  >>> print(eval("A → (¬B ∨ C)", {"A":True, "B": False, "C": False}))
  True
  >>> print(eval("A ∧ (X ∧ ¬Y)", {"A":True, "X": False, "Y": True}))
  False
  """
  return None

import doctest
doctest.run_docstring_examples(eval, globals())

# Exercise 4

Recall the language for intervals from Exercise 2.

## Exercise 4.1

Write a `Lark` grammar to obtain the AST for interval ranges.

In [ ]:
from lark import Lark, Transformer

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser('[-1.5 ; +2] [3 ; +4.7]')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), []), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '-1.5')]), Tree(Token('RULE', 'number'), [Token('INT', '+2')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '+4.7')])])])])

  >>> run_parser("[3 ; 3.5] [5 ; 12] [48 ; 73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), []), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '5')]), Tree(Token('RULE', 'number'), [Token('INT', '12')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '48')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '73.2')])])])])

  >>> run_parser("[3 ; 3.5] [1 ; 12] [48 ; 73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), []), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '1')]), Tree(Token('RULE', 'number'), [Token('INT', '12')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '48')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '73.2')])])])])

  >>> run_parser("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'intervals'), []), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '2')]), Tree(Token('RULE', 'number'), [Token('INT', '1')])])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '0.5')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '-73.2')])])])])
  """
  grammar = None
  parser = Lark(grammar)
  return parser.parse(data)

doctest.run_docstring_examples(run_parser, globals())

## Exercise 4.2

Write a `Lark` transformer to obtain obtain the number of intervals and overall range.

In [ ]:
def stats(data):
  """
  Lark statististics collector with doctests.

  >>> print(stats('[-1 ; 0] [0 ; 1] [2 ; 3] [3 ; 4]'))
  (4, 4.0)
  >>> print(stats('[-200 ; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  (4, 182.2)
  >>> print(stats("[3 ; 3.5] [5 ; 12] [48 ; 73.2]"))
  (3, 32.7)
  >>> print(stats("[34 ; 36.5]"))
  (1, 2.5)
  >>> print(stats("[3 ; 3.5] [1 ; 12] [48 ; 73.2]"))
  (3, 36.7)
  >>> print(stats("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  (3, -74.2)
  """
  return None

doctest.run_docstring_examples(stats, globals())

## Exercise 4.3

Write a `Lark` transformer to validate of intervals (always increasing or always decreasing)

In [ ]:
def validate(data):
  """
  Lark interval validator with doctests.

  >>> print(validate('[-1 ; 0] [0 ; 1] [2 ; 3] [3 ; 4]'))
  True
  >>> print(validate('[-200 ; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  True
  >>> print(validate("[3 ; 3.5] [5 ; 12] [48 ; 13.2]"))
  False
  >>> print(validate("[34 ; 36.5]"))
  True
  >>> print(validate("[3 ; 3.5] [63 ; 12] [48 ; 73.2]"))
  False
  >>> print(validate("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  False
  """
  return None

doctest.run_docstring_examples(validate, globals())

# Exercise 5

Consider a tiny functional language that supports:

- Integer and Boolean literals
- Binary operators (`+`, `*`, `<`, `and`, `or`)
- Conditional expressions (`if ... then ... else ...`)

For instance, the following is a valid expression in this language:
```
if 2 > 3 or 4 < 9 then 10 else -1*10
```

## Exercise 5.1

Write a `lark` grammar to obtain the AST for this tiny language.

In [ ]:
def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser('if 2 > 3 or 4 < 9 then 10 else -1*10')
  Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'comp'), [Token('INT', '2'), Token('GT', '>'), Token('INT', '3')]), Token('OR', 'or'), Tree(Token('RULE', 'comp'), [Token('INT', '4'), Token('LT', '<'), Token('INT', '9')])]), Token('INT', '10'), Tree(Token('RULE', 'arithterm'), [Token('INT', '-1'), Token('MUL', '*'), Token('INT', '10')])])

  >>> run_parser('if 2 > 3 or 4 then 10 else -1*10')
  Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'comp'), [Token('INT', '2'), Token('GT', '>'), Token('INT', '3')]), Token('OR', 'or'), Token('INT', '4')]), Token('INT', '10'), Tree(Token('RULE', 'arithterm'), [Token('INT', '-1'), Token('MUL', '*'), Token('INT', '10')])])

  >>> run_parser('if True or False and 10 < 20 then 10 else -1*10*30')
  Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'disj'), [Token('BOOL', 'True'), Token('OR', 'or'), Tree(Token('RULE', 'conj'), [Token('BOOL', 'False'), Token('AND', 'and'), Tree(Token('RULE', 'comp'), [Token('INT', '10'), Token('LT', '<'), Token('INT', '20')])])]), Token('INT', '10'), Tree(Token('RULE', 'arithterm'), [Token('INT', '-1'), Token('MUL', '*'), Token('INT', '10'), Token('MUL', '*'), Token('INT', '30')])])

  >>> run_parser('if True then 10 else -1*(10 or 20)')
  Tree(Token('RULE', 'ifthenelse'), [Token('BOOL', 'True'), Token('INT', '10'), Tree(Token('RULE', 'arithterm'), [Token('INT', '-1'), Token('MUL', '*'), Tree(Token('RULE', 'disj'), [Token('INT', '10'), Token('OR', 'or'), Token('INT', '20')])])])

  >>> run_parser('if 2 > 3 then 10 else True')
  Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'comp'), [Token('INT', '2'), Token('GT', '>'), Token('INT', '3')]), Token('INT', '10'), Token('BOOL', 'True')])

  >>> run_parser('if 2 > 3 then if True then 10 else 20 else 30')
  Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'comp'), [Token('INT', '2'), Token('GT', '>'), Token('INT', '3')]), Tree(Token('RULE', 'ifthenelse'), [Token('BOOL', 'True'), Token('INT', '10'), Token('INT', '20')]), Token('INT', '30')])

  >>> run_parser('if False then if True then 10 else 20 else if True then 20 else 50')
  Tree(Token('RULE', 'ifthenelse'), [Token('BOOL', 'False'), Tree(Token('RULE', 'ifthenelse'), [Token('BOOL', 'True'), Token('INT', '10'), Token('INT', '20')]), Tree(Token('RULE', 'ifthenelse'), [Token('BOOL', 'True'), Token('INT', '20'), Token('INT', '50')])])

  >>> run_parser('if if True then False else True then if True then 10 else 20 else if True then 20 else 50')
  Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'ifthenelse'), [Token('BOOL', 'True'), Token('BOOL', 'False'), Token('BOOL', 'True')]), Tree(Token('RULE', 'ifthenelse'), [Token('BOOL', 'True'), Token('INT', '10'), Token('INT', '20')]), Tree(Token('RULE', 'ifthenelse'), [Token('BOOL', 'True'), Token('INT', '20'), Token('INT', '50')])])

  >>> run_parser('20')
  Token('INT', '20')
  """
  grammar = None
  parser = Lark(grammar)
  return parser.parse(data)

doctest.run_docstring_examples(run_parser, globals())

## Exercise 5.2

Write a `lark` transformer to type check this language.

For instance, the following expression type checks:
```
if 2 > 3 or 4 < 9 then 10 else -1*10
```

But the following (syntactically correct) should throw a type error:

```
if 2 > 3 or 4 then 10 else -1*10
```
```
if 2 > 3 or 4 < 10 then 10 else -1*(10 or 20)
```
```
if 2 > 3 then 10 else True
```


In [ ]:
def type_check(data):
  """
  Lark type checker with doctests.

  >>> print(type_check('if 2 > 3 or 4 < 9 then 10 else -1*10'))
  <class 'int'>

  >>> print(type_check('if 2 > 3 or 4 then 10 else -1*10'))
  Type error in or

  >>> print(type_check('if True or False and 10 < 20 then 10 else -1*10*30'))
  <class 'int'>

  >>> print(type_check('if True then 10 else -1*(10 and 20)'))
  Type error in and

  >>> print(type_check('if 2 > 3 then 10 else True'))
  Type error in if-then-else

  >>> print(type_check('if 2 > 3 then if True then 10 else 20 else 30'))
  <class 'int'>

  >>> print(type_check('if False then if True then 10 else 20 else if True then 20 else 50'))
  <class 'int'>

  >>> print(type_check('if 10 < 20 then 20 < 10 else 20 < 23'))
  <class 'bool'>

  >>> print(type_check('if if True then False else True then if True then 10 else 20 else if True then 20 else 50'))
  <class 'int'>
  """
  return None

doctest.run_docstring_examples(type_check, globals())

-- nmm, 2026-01-10